In [28]:
import os
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm   # Acrescentado tqdm para barra de progresso
from calendar import monthrange # Cria uma lista de tuplas (mês, dia) apenas para datas válidas

In [29]:
path_dados = 'p1547014.txt'
path_estacao = 'precip.txt'

df_dados = pd.read_csv(path_dados, sep=';', dtype=str)
df_precip = pd.read_csv(path_estacao, sep=',', dtype=str)

In [30]:
df_precip

,ID,NAME,LAT,LONG,ELEVATION
0,0,p1547014,-15.9792,-47.975,1206


In [31]:
df_dados

,19670101
0,28.0
1,21.3
2,26.0
3,10.6
4,11.9
...,...
20814,0.1
20815,5.2
20816,0.0
20817,13.5


In [32]:
nrow = df_dados.shape[0]

In [33]:
primeira_data = df_dados.columns[0]

In [34]:
primeira_data


'19670101'

In [35]:
# Converter para datetime
dt = datetime.strptime(primeira_data, '%Y%m%d')

# Gerar lista de datas
datas = [dt + timedelta(days=i) for i in range(nrow)]

# Converter para ano juliano
anos_julianos = [f"{d.year}{d.timetuple().tm_yday:03d}" for d in datas]

# Se quiser adicionar ao seu DataFrame já existente:
df_julianio = pd.DataFrame({'ano_juliano': anos_julianos})
print(df_julianio)

      ano_juliano
0         1967001
1         1967002
2         1967003
3         1967004
4         1967005
...           ...
20814     2023361
20815     2023362
20816     2023363
20817     2023364
20818     2023365

[20819 rows x 1 columns]


In [ ]:
df_geral = pd.concat([df_julianio, df_dados], axis=1)
df_geral

,ano_juliano,19670101
0,1967001,28.0
1,1967002,21.3
2,1967003,26.0
3,1967004,10.6
4,1967005,11.9
...,...,...
20814,2023361,0.1
20815,2023362,5.2
20816,2023363,0.0
20817,2023364,13.5


In [64]:
def gerar_pcp1_pcp(df_estacao, df_dados, nome_arquivo='pcp1.pcp'):
    
    # Chegar se o df_estacao e df_dados não estão vazios
    if df_estacao.empty or df_dados.empty:
        raise ValueError("DataFrames de estação ou dados estão vazios.")
    
        
    # Quantidade de linhas do DataFrame de dados
    nrow = df_dados.shape[0]
    
    # Primeira data 
    primeira_data = df_dados.columns[0]
    
    # Converter para datetime
    dt = datetime.strptime(primeira_data, '%Y%m%d')

    # Gerar lista de datas
    datas = [dt + timedelta(days=i) for i in range(nrow)]

    # Converter para ano juliano
    anos_julianos = [f"{d.year}{d.timetuple().tm_yday:03d}" for d in datas]

    # Se quiser adicionar ao seu DataFrame já existente:
    df_julianio = pd.DataFrame({'ano_juliano': anos_julianos})
    
    # Concatenar DataFrames
    df_final = pd.concat([df_julianio, df_dados], axis=1)
    
    with open(nome_arquivo, 'w') as f:
        # Dados das estações
        nm_estacao = str(df_estacao['NAME'].values[0])
        lat_estacao = round(float(df_estacao['LAT'].values[0]),1)
        long_estacao = round(float(df_estacao['LONG'].values[0]),1)
        elev_estacao = int(df_estacao['ELEVATION'].values[0])

        f.write(f"Station  {nm_estacao}\n")
        f.write(f"Lati   {lat_estacao}\n")
        f.write(f"Long   {long_estacao}\n")
        f.write(f"Elev    {elev_estacao}\n")
        # Dados de chuva
        for _, row in tqdm(df_final.iterrows(), desc="Escrevendo dados de chuva"):
            # Formatar a linha de dados da chuva com 2 dígitos para o dia
            valor = float(row['19670101'])
            if valor == -99.0:
                f.write(f"{row['ano_juliano']}{float(row['19670101']):.1f}\n")
            elif valor > 99.9:
                f.write(f"{row['ano_juliano']}{float(row['19670101']):.1f}\n")
            elif valor == 0.0:
                f.write(f"{row['ano_juliano']}000.0\n")
            elif valor < 10.0:
                f.write(f"{row['ano_juliano']}00{float(row['19670101']):.1f}\n")
            else:
                f.write(f"{row['ano_juliano']}0{float(row['19670101']):.1f}\n")

    print(f"Arquivo {nome_arquivo} gerado com sucesso.")

In [65]:
gerar_pcp1_pcp(df_precip, df_dados, nome_arquivo='pcp1.pcp')

Escrevendo dados de chuva: 20819it [00:00, 55606.42it/s]

Arquivo pcp1.pcp gerado com sucesso.


In [ ]:
def gerar_pcp1_pcp(ano, mes, dia_inicial, dia_final, caminho_entrada, caminho_saida):
    # Verifica se o diretório de saída existe, se não, cria
    if not os.path.exists(caminho_saida):
        # os.makedirs(caminho_saida)
        raise NameError("O caminho não existe! Verifique o caminho de saída.")

    # Gera a lista de dias válidos no mês
    dias_validos = [dia for dia in range(dia_inicial, dia_final + 1) if dia <= monthrange(ano, mes)[1]]

    # Loop através dos dias válidos com barra de progresso
    for dia in tqdm(dias_validos, desc="Processando dias"):
        data_str = f"{ano:04d}{mes:02d}{dia:02d}"
        nome_arquivo = f"pcp1.pcp.{data_str}"
        caminho_arquivo = os.path.join(caminho_entrada, nome_arquivo)

        if os.path.isfile(caminho_arquivo):
            try:
                # Lê o arquivo CSV
                df = pd.read_csv(caminho_arquivo)

                # Verifica se as colunas necessárias existem
                if 'latitude' in df.columns and 'longitude' in df.columns and 'precipitation' in df.columns:
                    # Agrupa por latitude e longitude e soma a precipitação
                    df_agrupado = df.groupby(['latitude', 'longitude'], as_index=False)['precipitation'].sum()

                    # Salva o arquivo processado
                    nome_arquivo_saida = f"pcp1_processed_{data_str}.csv"
                    caminho_arquivo_saida = os.path.join(caminho_saida, nome_arquivo_saida)
                    df_agrupado.to_csv(caminho_arquivo_saida, index=False)
                else:
                    print(f"Colunas necessárias não encontradas em {nome_arquivo}. Pulando este arquivo.")
            except Exception as e:
                print(f"Erro ao processar {nome_arquivo}: {e}")
        else:
            print(f"Arquivo {nome_arquivo} não encontrado. Pulando este arquivo.")